In [9]:
# ============================================================================
# 环境检查：打印 PyTorch 与 tiktoken 的版本号。
# 版本很关键——tokenizer 的 BPE 词表、张量 API 都可能随版本变化。
# ============================================================================
from importlib.metadata import version

print('torch version:',version('torch'))
print('tiktoken version:',version('tiktoken'))

torch version: 2.11.0
tiktoken version: 0.14.0


In [ ]:
# 【第一步：文本分词 tokenizing】
# 目标：把连续的自然语言字符串切成一个个「词元(token)」。
# 语言模型无法直接处理原始字符串，必须先离散化成词元、再映射成整数 id。
#tokenizing text


In [10]:
# 读取原始训练语料：Edith Wharton 的短篇《The Verdict》纯文本。
# raw_text 是一个完整的 Python 字符串；打印字符总数与前 99 个字符做 sanity check。
with open("the-verdict.txt","r",encoding='utf-8') as f:
    raw_text=f.read()
print("Total number of character:", len(raw_text))
print(raw_text[:99])

Total number of character: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


In [11]:
# 用正则表达式做「基于规则」的分词。
# 分组括号 (...) 让 re.split 把分隔符本身也保留在结果里，
# 于是标点、空格、破折号 -- 都成为独立词元，而不会被丢弃。
# [item for ... if item] 过滤掉 split 产生的空字符串。
# 注意：这是最朴素的分词，仅用于教学，真实 LLM 用的是下面的 BPE。
import re
preprocessed=re.split(r'([,.:;?_!"()\']|--|\s)',raw_text)
preprocessed=[item for item in preprocessed if item]
print(preprocessed[:38])

['I', ' ', 'HAD', ' ', 'always', ' ', 'thought', ' ', 'Jack', ' ', 'Gisburn', ' ', 'rather', ' ', 'a', ' ', 'cheap', ' ', 'genius', '--', 'though', ' ', 'a', ' ', 'good', ' ', 'fellow', ' ', 'enough', '--', 'so', ' ', 'it', ' ', 'was', ' ', 'no', ' ']


In [12]:
# 统计分词后的词元总数（含重复）。这是「序列长度」量级的直观感受。
print("Number of tokens:", len(preprocessed))

Number of tokens: 8405


In [13]:
# set(...) 去重后取长度 = 语料中不同词元的种类数，即潜在词表规模。
len(set(preprocessed))

1132

In [14]:
# 对去重后的词元排序，得到确定性的词表顺序。
# 排序保证每次运行时 token -> id 的映射稳定可复现。
sorted(set(preprocessed))

['\n',
 ' ',
 '!',
 '"',
 "'",
 '(',
 ')',
 ',',
 '--',
 '.',
 ':',
 ';',
 '?',
 'A',
 'Ah',
 'Among',
 'And',
 'Are',
 'Arrt',
 'As',
 'At',
 'Be',
 'Begin',
 'Burlington',
 'But',
 'By',
 'Carlo',
 'Chicago',
 'Claude',
 'Come',
 'Croft',
 'Destroyed',
 'Devonshire',
 'Don',
 'Dubarry',
 'Emperors',
 'Florence',
 'For',
 'Gallery',
 'Gideon',
 'Gisburn',
 'Gisburns',
 'Grafton',
 'Greek',
 'Grindle',
 'Grindles',
 'HAD',
 'Had',
 'Hang',
 'Has',
 'He',
 'Her',
 'Hermia',
 'His',
 'How',
 'I',
 'If',
 'In',
 'It',
 'Jack',
 'Jove',
 'Just',
 'Lord',
 'Made',
 'Miss',
 'Money',
 'Monte',
 'Moon-dancers',
 'Mr',
 'Mrs',
 'My',
 'Never',
 'No',
 'Now',
 'Nutley',
 'Of',
 'Oh',
 'On',
 'Once',
 'Only',
 'Or',
 'Perhaps',
 'Poor',
 'Professional',
 'Renaissance',
 'Rickham',
 'Riviera',
 'Rome',
 'Russian',
 'Sevres',
 'She',
 'Stroud',
 'Strouds',
 'Suddenly',
 'That',
 'The',
 'Then',
 'There',
 'They',
 'This',
 'Those',
 'Though',
 'Thwing',
 'Thwings',
 'To',
 'Usually',
 'Venetian',


In [15]:
# 【第二步：把词元转换成词元 id (token IDs)】
# 建立「词表 vocab」：给每个唯一词元分配一个整数 id，模型内部只认整数。
#converting tokens into token IDs

In [16]:
# all_words：排好序的唯一词元列表；vocab_size：词表大小。
# 这里约 1132，即模型 embedding 层最终需要覆盖这么多个 id。
all_words=sorted(set(preprocessed))
vocab_size=len(all_words)
print(vocab_size)

1132


In [17]:
# 字典推导式构建 词表：{词元: 整数id}。
# enumerate 给排序后的每个词元一个从 0 递增的 id。
vocab={token:integer for integer,token in enumerate(all_words)}

In [18]:
# 浏览词表前 51 项，直观查看 token->id 的映射（如 '\n'->0, ' '->1 ...）。
for i ,item in enumerate(vocab.items()):
    print(item)
    if i>=50:
        break

('\n', 0)
(' ', 1)
('!', 2)
('"', 3)
("'", 4)
('(', 5)
(')', 6)
(',', 7)
('--', 8)
('.', 9)
(':', 10)
(';', 11)
('?', 12)
('A', 13)
('Ah', 14)
('Among', 15)
('And', 16)
('Are', 17)
('Arrt', 18)
('As', 19)
('At', 20)
('Be', 21)
('Begin', 22)
('Burlington', 23)
('But', 24)
('By', 25)
('Carlo', 26)
('Chicago', 27)
('Claude', 28)
('Come', 29)
('Croft', 30)
('Destroyed', 31)
('Devonshire', 32)
('Don', 33)
('Dubarry', 34)
('Emperors', 35)
('Florence', 36)
('For', 37)
('Gallery', 38)
('Gideon', 39)
('Gisburn', 40)
('Gisburns', 41)
('Grafton', 42)
('Greek', 43)
('Grindle', 44)
('Grindles', 45)
('HAD', 46)
('Had', 47)
('Hang', 48)
('Has', 49)
('He', 50)


In [22]:
# 【封装分词器 SimpleTokenizerV1】
# 一个最简分词器类，包含两个词表方向的映射与 encode/decode 两个方法：
#   str_to_int：词元 -> id（正向，用于 encode）
#   int_to_str：id -> 词元（反向，用于 decode）
# encode(text)：先用同样的正则切词、strip 去空白，再逐个查表转成 id 列表。
#   ⚠️ 局限：若文本含词表里没有的词(OOV)，str_to_int[s] 会直接 KeyError。
#   这正是后面要引入 BPE 的动机——BPE 靠子词永不 OOV。
# decode(ids)：把 id 逐个转回词元、用空格拼接，
#   再用 re.sub 把标点前多余的空格去掉，尽量还原自然文本。
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s,i in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.?_!"()\']|--|\s)', text)
        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # Replace spaces before the specified punctuations
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

In [23]:
# 实例化分词器并对一段测试文本编码。
# 观察输出的 ids 列表：每个整数对应词表里的一个词元。
tokenizer = SimpleTokenizerV1(vocab)
text=""""It's the last he painted, you know,"
           Mrs. Gisburn said with pardonable pride."""
ids=tokenizer.encode(text)
print(ids)

[3, 58, 4, 852, 990, 604, 535, 748, 7, 1128, 598, 7, 3, 69, 9, 40, 853, 1110, 756, 795, 9]


In [24]:
# 把 ids 解码回文本，检验 decode 的效果（注意标点/空格的还原细节）。
tokenizer.decode(ids)

'" It\' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.'

In [25]:
# encode 再 decode 的「往返(round-trip)」测试：
# 理想情况下应尽量还原原文，用来验证分词器的可逆性。
tokenizer.decode(tokenizer.encode(text))

'" It\' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.'

In [26]:
# 【第三步：字节对编码 BytePair Encoding (BPE)】
# 这是 GPT-2/3/4 真正使用的分词法。核心思想：从字节开始，反复合并
# 「最高频的相邻符号对」，得到介于字符与整词之间的「子词(subword)」单元。
# 优点：词表固定(GPT-2 为 50257)，却能表示任意字符串——生僻词被拆成子词，
# 永不 OOV，同时常见词仍是单个 token，兼顾覆盖率与效率。
#BytePair encoding

In [27]:
# 打印 tiktoken 版本，确认 BPE 库可用。
import importlib
import tiktoken
print("tiktoken version:", importlib.metadata.version("tiktoken"))

tiktoken version: 0.14.0


In [28]:
# 加载 GPT-2 预训练好的 BPE 分词器（自带 50257 词表与合并规则）。
tokenizer=tiktoken.get_encoding('gpt2')

In [29]:
# 用 BPE 编码一段含特殊 token 的文本。
# <|endoftext|> 是 GPT-2 的特殊分隔符：用于标记文档边界、拼接多篇语料。
# allowed_special 显式允许它被识别为「单个 id」(50256)，而不是被拆成普通字符。
# 观察：someunknownPlace 这类生僻词会被 BPE 拆成多个子词 id，体现「不 OOV」。
text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces"
     "of someunknownPlace."
)
integers=tokenizer.encode(text,allowed_special={"<|endoftext|>"})
print(integers)

[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 1659, 617, 34680, 27271, 13]


In [31]:
# 解码回文本，验证 BPE 的可逆性（BPE 是无损的，能完美还原）。
strings=tokenizer.decode(integers)
print(strings)

Hello, do you like tea? <|endoftext|> In the sunlit terracesof someunknownPlace.


In [32]:
# 对一个「无意义字符串」编码：说明 BPE 对任意输入都能切成子词 id，绝不失败。
tokenizer.encode("Akwirw ier", allowed_special={"<|endoftext|>"})

[33901, 86, 343, 86, 220, 959]

In [33]:
# 往返测试：证明即便是随机字符串，encode->decode 也能精确还原原文。
print(tokenizer.decode(tokenizer.encode("Akwirw ier", allowed_special={"<|endoftext|>"})))

Akwirw ier


In [ ]:
# 【第四步：用滑动窗口做数据采样，构造训练样本】
# GPT 的训练目标是「预测下一个 token」(next-token prediction)。
# 因此需要成对的 (input, target)，其中 target 是 input 整体右移一位。
#Data sampling with a sliding window

In [ ]:
# 从配套模块导入数据集类与数据加载工厂函数（定义见 supplementary.py）。
from supplementary import GPTDatasetV1, create_dataloader_v1

In [ ]:
# 调用工厂函数构造 DataLoader（现为模块级函数，直接调用即可）。
# 参数含义：
#   batch_size=8  每批 8 条样本
#   max_length=4  每条样本长度为 4 个 token（上下文窗口）
#   stride=4      窗口步长=4，与 max_length 相等 -> 相邻样本不重叠
#   shuffle=False 保持顺序，便于观察输入/目标的对应关系
# iter + next 取出第一个 batch。
# 关键观察：targets 的每一行都等于 inputs 对应行右移一位——
# 即模型看到 inputs[t] 时要预测出 targets[t]，这就是 next-token 训练信号。
# inputs/targets 形状均为 (batch_size, max_length) = (8, 4)。

dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=4, stride=4, shuffle=False)

data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Inputs:\n", inputs)
print("\nTargets:\n", targets)